# Convert (nb1): Download DICOM → NIfTI  —  shared, model-agnostic

First step of the harmonized Segmentator workflow (runs on the GPU VM before the model's
inference notebook). It:
1. Downloads DICOM series from Imaging Data Commons (IDC) **or** a private GCS bucket.
2. Converts each series to NIfTI via `dcm2niix`.
3. Emits the **Boundary-A** archive `converted_nifti.tar.lz4` with a canonical layout:
   `<SeriesInstanceUID>/<SeriesInstanceUID>.nii.gz`, plus `convert_manifest.json`.

The model inference notebook (nb2) consumes only `converted_nifti.tar.lz4` — it never sees
raw DICOM — so this notebook is identical for every model.

## Imports

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import yaml
from idc_index.index import IDCClient

NOTEBOOK_START = time.time()

def _elapsed(since=None):
    return f"{time.time() - (since if since is not None else NOTEBOOK_START):.1f}s"

def _dir_size_mb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / (1024 ** 2)

print(f"Python      : {sys.version}")
print(f"Working dir : {os.getcwd()}")
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

Tagged `parameters` so papermill injects values at runtime (the WDL passes
`-y SeriesInstanceUIDs`, `-p input_uri`, `-p secret_project`).

In [ ]:
# Papermill injects SeriesInstanceUIDs as a Python list via `-y`.
SeriesInstanceUIDs = [
    "1.3.6.1.4.1.14519.5.2.1.7009.9004.100143549999116733615345241533"
]

# Empty → download from IDC using SeriesInstanceUIDs (default).
# gs:// URI → download everything under that path and sort by SeriesInstanceUID tag.
input_uri = ""

# GCP project holding the s5cmd HMAC secrets. Only needed when input_uri is set.
secret_project = ""

## Normalize parameters

In [ ]:
def _flatten_uids(raw):
    """Normalize any papermill input shape (str / list / dict) to a clean list of UIDs.
    Tolerates malformed YAML where UIDs are whitespace- rather than comma-separated."""
    if isinstance(raw, str):
        try:
            parsed = yaml.safe_load(raw)
        except Exception:
            parsed = raw
        if isinstance(parsed, dict):
            parsed = parsed.get('SeriesInstanceUIDs', list(parsed.values()))
        raw = parsed
    items = list(raw) if isinstance(raw, (list, tuple)) else [raw]
    flat = []
    for item in items:
        if item is None:
            continue
        flat.extend(p for p in re.split(r'[\s,]+', str(item).strip()) if p)
    cleaned = [u for u in flat if re.fullmatch(r'[\d.]+', u)]
    seen, deduped = set(), []
    for u in cleaned:
        if u not in seen:
            seen.add(u)
            deduped.append(u)
    return deduped

if input_uri:
    if not input_uri.startswith('gs://'):
        raise ValueError(f'input_uri must start with gs:// — got {input_uri!r}')
    if not secret_project:
        raise ValueError('secret_project must be set when input_uri is set')
    series_uids = []
else:
    series_uids = _flatten_uids(SeriesInstanceUIDs)
    if not series_uids:
        raise ValueError(f'No valid SeriesInstanceUIDs parsed from input: {SeriesInstanceUIDs!r}')

DICOM_DIR = Path('/tmp/dicom')
NIFTI_DIR = Path('/tmp/converted_nifti')
for _d in (DICOM_DIR, NIFTI_DIR):
    _d.mkdir(parents=True, exist_ok=True)

usage_metrics = {'series': {}}
if input_uri:
    print(f'Input URI : {input_uri}  (series discovered after download)')
else:
    print(f'Series to process : {len(series_uids)}')
    for u in series_uids:
        print(f'  {u}')

## Download DICOM from IDC

In [ ]:
download_errors = []
sort_errors = []

if not input_uri:
    client = IDCClient()
    print(f'[T+{_elapsed()}] Starting download of {len(series_uids)} series')
    for uid in series_uids:
        dest = DICOM_DIR / uid
        dest.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        print(f'  Downloading {uid} ...', flush=True)
        try:
            client.download_from_selection(downloadDir=str(dest), seriesInstanceUID=uid)
            elapsed = round(time.time() - t0, 1)
            dcm_files = list(dest.rglob('*.dcm'))
            size_mb = _dir_size_mb(dest)
            usage_metrics['series'].setdefault(uid, {})
            usage_metrics['series'][uid].update(
                download_s=elapsed, download_dcm_files=len(dcm_files), download_mb=round(size_mb, 1))
            print(f'  Downloaded {uid}: {len(dcm_files)} files  {size_mb:.1f} MB  in {elapsed}s')
        except Exception as exc:
            download_errors.append(f'{uid}: {traceback.format_exc()}')
            print(f'  ERROR downloading {uid}: {exc}')
    if download_errors:
        Path('download_error_file.txt').write_text('\n'.join(download_errors))
    print(f'[T+{_elapsed()}] Download phase complete ({len(download_errors)} failed)')

## Download DICOM from GCS

Used when `input_uri` is set. Stages everything under `input_uri` with `s5cmd`
(HMAC keys from Secret Manager), then sorts files into `DICOM_DIR/<uid>/` by
reading each file's SeriesInstanceUID tag.

In [ ]:
if input_uri:
    bucket, _, prefix = input_uri[len('gs://'):].partition('/')
    print(f'[T+{_elapsed()}] Fetching HMAC credentials (project={secret_project})')
    from google.cloud import secretmanager
    _sm = secretmanager.SecretManagerServiceClient()
    def _secret(sid):
        name = f'projects/{secret_project}/secrets/{sid}/versions/latest'
        return _sm.access_secret_version(name=name).payload.data.decode('utf-8').strip()
    Path('~/.aws').expanduser().mkdir(parents=True, exist_ok=True)
    Path('~/.aws/credentials').expanduser().write_text(
        f"[default]\naws_access_key_id = {_secret('s5cmd-hmac-key-id')}\n"
        f"aws_secret_access_key = {_secret('s5cmd-hmac-secret')}\n")
    staging = Path('/tmp/dicom_staging'); staging.mkdir(parents=True, exist_ok=True)
    s3_path = f's3://{bucket}/{prefix}/*' if prefix else f's3://{bucket}/*'
    print(f'[T+{_elapsed()}] s5cmd cp {s3_path} -> {staging}', flush=True)
    subprocess.run(['s5cmd', '--endpoint-url', 'https://storage.googleapis.com',
                    'cp', s3_path, str(staging) + '/'], check=True)
    import pydicom
    for dcm_file in staging.rglob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(dcm_file), stop_before_pixels=True)
            dest = DICOM_DIR / str(ds.SeriesInstanceUID)
            dest.mkdir(parents=True, exist_ok=True)
            shutil.move(str(dcm_file), str(dest / dcm_file.name))
        except Exception as exc:
            sort_errors.append(f'{dcm_file}: {exc}')
    shutil.rmtree(staging)
    series_uids = sorted({d.name for d in DICOM_DIR.iterdir() if d.is_dir()})
    for uid in series_uids:
        dest = DICOM_DIR / uid
        usage_metrics['series'].setdefault(uid, {})
        usage_metrics['series'][uid].update(
            download_dcm_files=len(list(dest.rglob('*.dcm'))), download_mb=round(_dir_size_mb(dest), 1))
    if sort_errors:
        Path('download_error_file.txt').write_text('\n'.join(sort_errors))
    print(f'[T+{_elapsed()}] Sorted into {len(series_uids)} series ({len(sort_errors)} file(s) failed)')

## Convert DICOM → NIfTI (Boundary-A layout)

Standardized layout `<uid>/<uid>.nii.gz`. `dcm2niix` may emit several NIfTIs for a
series (e.g. localizers, derived); the largest volume is chosen as the primary and
renamed to `<uid>.nii.gz`. Any secondary volumes are kept alongside as
`<uid>__<original>.nii.gz` so nothing is silently dropped.

In [ ]:
dcm2niix_errors = []
converted = {}
failed_dl = {e.split(':')[0] for e in download_errors}
to_convert = [u for u in series_uids if u not in failed_dl]
print(f'[T+{_elapsed()}] Converting {len(to_convert)} series')

for uid in to_convert:
    dcm_path = DICOM_DIR / uid
    out_dir = NIFTI_DIR / uid
    work = out_dir / '_raw'
    work.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    result = subprocess.run(
        ['dcm2niix', '-z', 'y', '-f', '%s_%d', '-o', str(work), str(dcm_path)],
        capture_output=True, text=True)
    nii_files = sorted(work.glob('*.nii.gz'), key=lambda f: f.stat().st_size, reverse=True)
    if result.returncode != 0 or not nii_files:
        dcm2niix_errors.append(f'{uid}: rc={result.returncode}\n{result.stderr}')
        print(f'  ERROR converting {uid} (rc={result.returncode})')
        shutil.rmtree(work, ignore_errors=True)
        continue
    primary = out_dir / f'{uid}.nii.gz'
    shutil.move(str(nii_files[0]), str(primary))
    extras = []
    for f in nii_files[1:]:
        tgt = out_dir / f'{uid}__{f.name}'
        shutil.move(str(f), str(tgt))
        extras.append(tgt.name)
    shutil.rmtree(work, ignore_errors=True)
    converted[uid] = {'primary': primary.name, 'extras': extras}
    usage_metrics['series'].setdefault(uid, {})['dcm2niix_s'] = round(time.time() - t0, 1)
    print(f'  Converted {uid} -> {primary.name}' + (f'  (+{len(extras)} extra)' if extras else ''))

if dcm2niix_errors:
    Path('dcm2niix_error_file.txt').write_text('\n'.join(dcm2niix_errors))
print(f'[T+{_elapsed()}] Conversion complete ({len(converted)}/{len(to_convert)} succeeded)')

## Package Boundary-A archive + manifest

In [ ]:
manifest = {
    'boundary': 'A',
    'layout': '<SeriesInstanceUID>/<SeriesInstanceUID>.nii.gz',
    'input_source': input_uri if input_uri else 'IDC',
    'series': {uid: converted[uid] for uid in converted},
    'series_count': len(converted),
}
(NIFTI_DIR / 'convert_manifest.json').write_text(json.dumps(manifest, indent=2))

if not converted:
    raise RuntimeError('No series converted to NIfTI — see download/dcm2niix error files')

cmd = f'tar -cf - -C {NIFTI_DIR.parent} {NIFTI_DIR.name} | lz4 > converted_nifti.tar.lz4'
res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError(f'Compression failed: {res.stderr}')
size_mb = Path('converted_nifti.tar.lz4').stat().st_size / (1024 ** 2)
print(f'[T+{_elapsed()}] Wrote converted_nifti.tar.lz4 ({size_mb:.1f} MB, {len(converted)} series)')

## Usage metrics + summary

In [ ]:
import csv
usage_metrics['total_elapsed_s'] = round(time.time() - NOTEBOOK_START, 1)
with open('convert_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'download_s', 'download_dcm_files', 'download_mb',
                'dcm2niix_s', 'run_total_elapsed_s'])
    for uid, m in usage_metrics['series'].items():
        w.writerow([uid, m.get('download_s', ''), m.get('download_dcm_files', ''),
                    m.get('download_mb', ''), m.get('dcm2niix_s', ''),
                    usage_metrics['total_elapsed_s']])

print('=' * 60)
print('Convert (nb1) Summary')
print('=' * 60)
print(f'  Input source      : {input_uri if input_uri else "IDC"}')
print(f'  Series converted  : {len(converted)}')
print(f'  Download failures : {len(download_errors) + len(sort_errors)}')
print(f'  dcm2niix failures : {len(dcm2niix_errors)}')
print(f'  Total wall-clock  : {usage_metrics["total_elapsed_s"]}s')
print('=' * 60)